In [ ]:
import os
import sys
from tqdm import tqdm
import glob
import typing
import import_ipynb

# Add current directory to path for imports
import os
current_dir = "/home2/ducvu/speech-processing-implement/codes"
if current_dir not in sys.path:
    sys.path.insert(0, current_dir)

import importlib
import config
from config import *
config_classifiers = config

import numpy as np
import pandas as pd
from collections import defaultdict

import subprocess
import soundfile as sf
import torch
from pathlib import Path

In [ ]:
input_file = "/home2/ducvu/speech-processing-implement/raw_data/sub-1/participant_001_sub.wav"
output_file = "/home2/ducvu/speech-processing-implement/raw_data/sub-1/participant_001_sub_normalize_channel.wav"

In [ ]:
def normalize_channels_sox(input_file, output_file="normalized.wav",
                          sample_rate=16000,
                          lowpass_freq=3400,
                          highpass_freq=200,
                          compression_factor=8,
                          keep_intermediate=False):

    # Create intermediate file paths
    base_name = Path(input_file).stem
    temp_dir = Path("temp_sox")
    temp_dir.mkdir(exist_ok=True)
    
    gsm_file = temp_dir / f"{base_name}_gsm.gsm"

    sox_encode_cmd = [
        'sox',
        input_file,
        '-r', str(sample_rate),        # Resample to 16kHz
        '-c', '1',                      # Convert to mono
        '-e', 'gsm-full-rate',         # GSM FULL RATE encoding (13 kbps)
        str(gsm_file),
        'compand',                      # Compression
        '0.3,1',                        # Attack, decay
        f'6:-70,-60,-20',              # Transfer function
        '-5',                           # Soft knee dB
        '-90',                          # Gain
        '0.2'                           # Initial volume
    ]

    result = subprocess.run(sox_encode_cmd, 
                            capture_output=True, 
                            text=True, 
                            check=True)
    print(f"  ✓ GSM encoding complete: {gsm_file}")


    sox_decode_cmd = [
        'sox',
        str(gsm_file),
        '-b', '16',                     # 16-bit encoding
        '-e', 'signed-integer',         # Signed integer
        output_file,
        'highpass', str(highpass_freq), # High-pass filter (200 Hz)
        'lowpass', str(lowpass_freq),   # Low-pass filter (3400 Hz)
        'rate', str(sample_rate)        # Ensure 16kHz
    ]

    result = subprocess.run(sox_decode_cmd, 
                        capture_output=True, 
                        text=True, 
                        check=True)

    # Clean up intermediate files
    if not keep_intermediate:
        gsm_file.unlink()
        if not any(temp_dir.iterdir()):
            temp_dir.rmdir()
    
    print(f"\n✓ Channel normalization complete: {output_file}")
    return output_file

In [ ]:
normalize_channels_sox(input_file, output_file)